In [1]:
# 랭체인에서 RAG (검색 + 생성) 구현, 분기체인 (규칙 기반 멀티 체인 선택)
!pip install langchain-google-genai google-genai langchain-chroma langchain-community
!pip install sentence-transformers python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.4 MB/s eta 0:00:00
  

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import CharacterTextSplitter # Changed import path
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()

# LLM 모델
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# 임베딩 모델
#embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001") # 과금됨
# 과금 없는 모델 사용
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 문서 데이터 준비
# 랭체인 RAG 구조는 문서들을 저장 -> 검색 -> 담변생성하는 흐름은 가짐
# 그런데 파일(.txt, .pdf, ...)을 읽으면 구조가 복잡해 일단 메모리에 문서데이터를 만든다. (Document type 사용)
# retriever는 'Document 단위' 검색을 함
docs = [
    Document(page_content="외야수 최원준(27)은 올 시즌 부진했다. KIA에서 타율 0.229에 그쳤다. 지난 7월 3 대 3 트레이드를 통해 NC로 이적한 뒤에도 반등하지 못했다. 프로 선수한테 가장 중요하다고도 할 수 있는 자유계약선수(FA) 직전 시즌을 타율 0.242에 6홈런으로 마쳤다. 여기에 FA A등급까지 받으면서, 최원준은 올겨울 큰 관심을 받기 어려울 것이라는 전망이 나왔다."),

    Document(page_content="NC는 2025시즌 외국인선수로 투수 2명, 타자 1명을 활용했다. 라일리 톰슨(29), 로건 앨런(28)으로 원투펀치를 꾸렸고, 맷 데이비슨(34)으로 중심타선을 채웠다. 3명의 선수 중 2명은 공수서 보탬이 됐다. 라일리는 한화 이글스의 코디 폰세(31)와 함께 17승으로 리그 다승 공동 선두에 올랐다. 데이비슨은 부상 여파에도 36홈런을 터트리며 리그 2위에 올랐다.")
]

# 텍스트를 조각으로 쪼개기 (옵션)
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
split_docs = text_splitter.split_documents(docs)

# 랭체인이 지원하는 Chromadb에 저장
db = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model
)

# Retriever
retriever = db.as_retriever()

# PromptTemplate 작성
prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("""
        너는 친절하고 똑똑한 AI 어시스턴트야.
        아래 문서 내용을 참고해서 나의 질문에 정확하게 답을 해줘.
        문서 내용이 불충분한 경우 '문서에 해당 정보가 없어요'라고 답변해.
        답변은 5행 정도만 좋아.
    """),
    HumanMessagePromptTemplate.from_template("""
        문맥:
        {context}
        질문:
        {question}
    """)
])

# 체인 생성 (LCEL 방식)

def format_docsFunc(docs):
    # 검색된 Document 리스트를 하나의 문자열로 합쳐 반환하는 헬퍼 함수
    return '\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {
        'context':retriever | format_docsFunc,  # 질문 -> retriever로 검색 -> 문서들을 문자열로 포맷
        'question':RunnablePassthrough()        # 원래 질문은 그대로 전달
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

# 질문
query = 'NC 소식을 좀 알려줄 수 있어?'
result = rag_chain.invoke(query)

print('질문: ', query)
print('답변: ', result)

/tmp/ipython-input-2683458568.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

질문:  NC 소식을 좀 알려줄 수 있어?
답변:  NC 다이노스 소식은 다음과 같습니다.

외야수 최원준은 지난 7월 트레이드로 NC에 이적했지만, 타율 0.242, 6홈런으로 부진하여 FA 시장에서 큰 관심을 받기 어려울 전망입니다.

2025시즌 외국인 선수로는 투수 라일리 톰슨과 로건 앨런, 타자 맷 데이비슨을 활용했습니다. 라일리 톰슨은 17승으로 리그 다승 공동 선두를 기록했고, 맷 데이비슨은 부상에도 불구하고 36홈런으로 리그 2위에 올랐습니다.


In [6]:
# 멀티 실행 체인: RunnableParallel을 사용해 여러 체인을 병렬로 처리
# 여건상 RunnableParalle 사용 불가. 그래서 여기서는 순차적으로 두 번 LLM 호출
llm_calm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
llm_creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)
q = "AI가 뭔가요?"
out1 = llm_calm.invoke(q)
out2 = llm_creative.invoke(q)
print("1) 현실적인 답변: ", out1.content)
print("2) 창의적인 답변: ", out2.content)


1) 현실적인 답변:  AI는 **인공지능(Artificial Intelligence)**의 줄임말입니다.

가장 간단하게 설명하자면, **컴퓨터 프로그램이나 기계가 인간처럼 생각하고, 배우고, 문제를 해결하고, 의사결정을 내리는 등의 지능적인 행동을 할 수 있도록 만드는 기술 또는 분야**입니다.

조금 더 자세히 설명해 드릴게요.

1.  **인간의 지능 모방:** AI의 핵심 목표는 인간의 인지 능력(학습, 추론, 문제 해결, 언어 이해, 시각 인지 등)을 기계가 흉내 내거나 더 나아가 능가하도록 만드는 것입니다.

2.  **어떻게 작동하나요?**
    *   **데이터 학습:** AI는 방대한 양의 데이터를 분석하여 패턴을 찾아내고, 그 패턴을 기반으로 스스로 학습합니다. 예를 들어, 수많은 고양이 사진을 보고 "이것이 고양이"라는 것을 학습하면, 처음 보는 고양이 사진도 고양이라고 인식할 수 있게 됩니다.
    *   **알고리즘:** 이러한 학습과 판단을 가능하게 하는 것이 바로 '알고리즘'입니다. 특히 **머신러닝(Machine Learning)**과 **딥러닝(Deep Learning)**이라는 기술이 AI를 구현하는 주요 방법입니다.

3.  **어떤 일을 할 수 있나요? (예시)**
    *   **음성 인식:** 스마트폰의 시리(Siri)나 빅스비(Bixby)처럼 사람의 말을 알아듣고 명령을 수행합니다.
    *   **이미지 인식:** 얼굴 인식 잠금 해제, 의료 영상 분석, 자율주행차의 주변 환경 인식 등에 사용됩니다.
    *   **추천 시스템:** 넷플릭스나 유튜브에서 시청 기록을 바탕으로 좋아할 만한 콘텐츠를 추천해 줍니다.
    *   **번역:** 구글 번역기처럼 다른 언어를 이해하고 번역해 줍니다.
    *   **자율주행:** 자동차가 스스로 주변을 인지하고 운전하는 기술입니다.
    *   **챗봇:** 고객 상담이나 정보 제공을 위해 사람과 대화하는 프로그램입니다.
    *   **콘텐츠 생성:** 최

In [7]:
print('\n\n분기 처리 (조건에 따라 체인을 선택)')
from langchain_core.runnables import RunnableBranch, RunnableLambda

print('RunnableBranch 이해: RunnableBranch((조건, 체인1), 체인2)')
def is_weather_question(text:str) -> bool:
    return '날씨' in text.lower() # 질문에 '날씨'란 단어가 포함되어 있으면 True, 아니면 False 반환

# 분기 a
weather_chain = RunnableLambda(lambda x: f'오늘의 날씨는 흐리고 기온은 12도입니다')

# 분기 b
general_chain = RunnableLambda(lambda x: f'일반적인 질문이군요. "{x}"에 대해 설명할게요')

# 분가 조합
branch_chain = RunnableBranch(
    (is_weather_question, weather_chain),
    general_chain
)

# 실행
print('날씨 질문 테스트: ', branch_chain.invoke('오늘 날씨 어때?'))
print('일반 질문 테스트: ', branch_chain.invoke('AI란 뭐니?'))



분기 처리 (조건에 따라 체인을 선택)
RunnableBranch 이해: RunnableBranch((조건, 체인1), 체인2)
날씨 질문 테스트:  오늘의 날씨는 흐리고 기온은 12도입니다
일반 질문 테스트:  일반적인 질문이군요. "AI란 뭐니?"에 대해 설명할게요


In [8]:
print('\n\n분기 처리 (LLM) 적용')
llm2 = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7)
from langchain_core.output_parsers import StrOutputParser

# prompt 생성 함수들 정의: 질문을 그대로 LLM에 던지지 않고 역할/형식을 지정한 후 질문하기

# 수학 질문용 프롬프트
def make_math_prompt(question:str) -> str:
    return(
        '너는 수학 풀이를 잘하는 모범생이야.\n'
        '아래 수학 문제를 단계별로 풀어 보고, 마지막 줄에 정답만 한 번 적어 줘.\n\n'
        f'문제: {question}\n\n'
        '풀이:'
    )

# 코딩 질문용 프롬프트
def make_code_prompt(question:str) -> str:
    return(
        '너는 친절한 프로그래밍 전문가야.\n'
        '아래 요청에 대해, 1) 간단한 설명 2) 예제 코드 3) 중요한 포인트 설명 순서로 답해줘.\n\n'
        f'문제: {question}\n\n'
        '답변:'
    )

# 일반 질문용 프롬프트
def make_general_prompt(question:str) -> str:
    return(
        '너는 유능한 데이터 과학자야.\n'
        '아래 질문에 대해 초보자도 이해할 수 있도록 5 ~ 6행 문장으로 설명해줘.\n\n'
        f'질문: {question}\n\n'
        '답변:'
    )

# 각 체인 구성 (LCEL 방식: 입력 프롬프트  -> LLM -> 결과출력)
math_chain = (
    RunnableLambda(make_math_prompt)
    | llm2
    | StrOutputParser()
)
code_chain = (
    RunnableLambda(make_code_prompt)
    | llm2
    | StrOutputParser()
)
general_chain = (
    RunnableLambda(make_general_prompt)
    | llm2
    | StrOutputParser()
)

# 분기 조건 (워크 플로우)
def is_math_question(text:str) -> bool:
    # 수학 관련 키워드 / 기호가 있으면 수학 질문으로 간주
    t = text.replace(' ', '').lower()
    math_keywords = ['더하기', '빼기', '곱하기', '나누기', '계산', '합', '치', '곱']
    math_symbols = ['+', '-', '*', '/', '^']
    return any(k in t for k in math_keywords) or any(s in t for s in math_symbols)

def is_code_question(text:str) -> bool:
    # 프로그래밍 관련 키워드 / 기호가 있으면 코딩 질문으로 간주
    t = text.lower()
    code_keywords = ['코드', '함수', '클래스', '메소드', '알고리즘', 'python', 'java', 'c언어']
    return any(k in t for k in code_keywords)

# 분기 처리 체인
branch_chain = RunnableBranch(
    (is_math_question, math_chain),
    (is_code_question, code_chain),
    general_chain
)

# 질문하기
q1 = '3 더하기 5 곱하기 2는 얼마인가?'
print('\n수학 질문 연습: ', q1)
print('결과 1: ', branch_chain.invoke(q1).strip())

q2 = '파이썬으로 숫자들의 평균을 구하는 예제를 만들어.'
print('\n코딩 질문 연습: ', q1)
print('결과 2: ', branch_chain.invoke(q2).strip())

q3 = '가을과 겨울의 차이를 설명해.'
print('\n일반 질문 연습: ', q1)
print('결과 3: ', branch_chain.invoke(q3).strip())



분기 처리 (LLM) 적용

수학 질문 연습:  3 더하기 5 곱하기 2는 얼마인가?
결과 1:  풀이:
주어진 문제는 덧셈과 곱셈이 함께 있는 식입니다. 수학에서는 연산의 우선순위가 있습니다. 곱셈과 나눗셈은 덧셈과 뺄셈보다 먼저 계산해야 합니다.

1.  **곱셈을 먼저 계산합니다.**
    5 곱하기 2 = 10

2.  **곱셈의 결과와 나머지 수를 더합니다.**
    3 더하기 10 = 13

정답: 13

코딩 질문 연습:  3 더하기 5 곱하기 2는 얼마인가?
결과 2:  안녕하세요! 숫자들의 평균을 구하는 건 데이터 분석의 아주 기본적인 단계예요. 파이썬으로 쉽게 할 수 있답니다.

1.  **평균의 정의:** 평균은 모든 숫자를 더한 후, 그 개수로 나누는 값입니다.
2.  **숫자 준비:** 파이썬에서는 먼저 숫자들을 `리스트(list)`에 담아요. 예를 들어 `[10, 20, 30]`처럼요.
3.  **합계 구하기:** `sum()` 함수를 사용하면 리스트 안의 모든 숫자를 간단하게 더할 수 있습니다.
4.  **개수 세기:** `len()` 함수는 리스트 안에 몇 개의 숫자가 있는지 개수를 세어줘요.
5.  **평균 계산:** 마지막으로, `sum()`으로 구한 합계를 `len()`으로 구한 개수로 나누면 평균이 나옵니다.

**예시:** `숫자들 = [10, 20, 30, 40]` 일 때, `sum(숫자들) / len(숫자들)`을 계산하면 평균인 25를 얻을 수 있어요!

일반 질문 연습:  3 더하기 5 곱하기 2는 얼마인가?
결과 3:  안녕하세요! 유능한 데이터 과학자의 눈으로 가을과 겨울의 차이를 명확하게 설명해 드릴게요.

가을과 겨울은 기온, 자연의 모습, 그리고 계절의 역할에서 뚜렷한 차이를 보입니다. 가을은 여름에서 겨울로 넘어가는 '환절기'로, 시원하고 쾌적한 기온이 특징입니다. 이때는 나무들이 알록달록 단풍으로 물들고 낙엽이 지면서 아름다운 풍경을 선사하죠.

반면 겨울은 한 해 중 가장 추운 '절정기'로, 기온